# EDA 009: Game catalog (table view)

## Key Goal

Render all indexed games in a compact table (name + app_id), matching the display style used in `recs_003`.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from cwd={here}")

REPO_ROOT = _repo_root()
INDEX_PATH = REPO_ROOT / "artifacts" / "recs" / "game_profile_embedding_index.parquet"
if not INDEX_PATH.is_file():
    raise FileNotFoundError(f"Missing index parquet: {INDEX_PATH}")

idx_df = pd.read_parquet(INDEX_PATH)
name_col = "app_name" if "app_name" in idx_df.columns else ("name" if "name" in idx_df.columns else None)
if name_col is None:
    raise KeyError(f"Could not find app-name column in index. Columns: {list(idx_df.columns)}")

games_catalog = (
    idx_df[["app_id", name_col]]
    .dropna(subset=[name_col])
    .drop_duplicates(subset=["app_id", name_col])
    .rename(columns={name_col: "app_name"})
    .sort_values("app_name", ignore_index=True)
)
print(f"{len(games_catalog)} games in index")


In [ ]:
chunk_size = 10

explanation_html = (
    "<div style='margin-bottom:8px;'>"
    "<b>Note:</b> The number shown directly under each game name is the game's <b>Steam app_id</b>—a unique identifier used internally on Steam and in this recommender."
    "</div>"
)

def make_row(chunk):
    return "<tr>" + "".join(
        f"<td>{row.app_name}<br><span style='color:gray; font-size:small;'>{row.app_id}</span></td>"
        for _, row in chunk.iterrows()
    ) + "".join("<td></td>" for _ in range(chunk_size - len(chunk))) + "</tr>"

rows_html = "\n".join(
    make_row(games_catalog.iloc[i:i + chunk_size])
    for i in range(0, len(games_catalog), chunk_size)
)

header = "".join([f"<th>Game {i + 1}</th>" for i in range(chunk_size)])
html_table = f"""
{explanation_html}
<table border='1' style='border-collapse:collapse'>
<thead><tr>{header}</tr></thead>
<tbody>
{rows_html}
</tbody>
</table>
"""
display(HTML(html_table))
